# Meridian — Backbone Pre-training (Colab)

Runs Phase 1 TCP self-supervised pre-training on the hybrid RA-GDN + SWA backbone.
Checkpoints save to Google Drive every 5 000 steps; re-run all cells to auto-resume
after a session drop.

**Run this before `colab_portfolio_return.ipynb`.**

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
HF_USERNAME   = 'Verdenroz'
HF_TOKEN      = ''            # HF token with read access (required if repos are private)
                              # create at: huggingface.co/settings/tokens
HF_MODEL_REPO = f'{HF_USERNAME}/meridian-checkpoints'
HF_DATA_REPO  = f'{HF_USERNAME}/meridian-data'

In [ ]:
# ── 1. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/meridian'
os.makedirs(f'{DRIVE_DIR}/checkpoints/pretrain', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/data/mmaps',           exist_ok=True)
print('Drive ready:', DRIVE_DIR)

In [ ]:
# ── 2. HuggingFace login + install deps ───────────────────────────────────────
import subprocess, sys
subprocess.run([
    'pip', 'install', '-q',
    'numpy>=2.1', 'pandas>=2.2', 'pyyaml>=6.0',
    'hmmlearn>=0.3', 'faiss-cpu>=1.8', 'tokenizers>=0.20',
    'huggingface_hub>=0.26',
], check=True)

from huggingface_hub import login, snapshot_download, hf_hub_download, HfApi
if HF_TOKEN:
    login(token=HF_TOKEN)
print('HuggingFace ready.')

In [ ]:
# ── 3. Download training code ─────────────────────────────────────────────────
import os, shutil

if not os.path.exists('/content/meridian'):
    local = snapshot_download(repo_id=HF_MODEL_REPO, allow_patterns='code/**')
    shutil.copytree(os.path.join(local, 'code'), '/content/meridian')
    print('Code downloaded.')
else:
    print('Code already present.')

In [ ]:
# ── 4. Download data (cached to Drive across sessions) ────────────────────────
import os, shutil

data_dir  = f'{DRIVE_DIR}/data/mmaps'
train_dir = f'{data_dir}/train'
has_data  = os.path.exists(train_dir) and any(f.endswith('.f32bin') for f in os.listdir(train_dir))

if not has_data:
    snapshot_download(repo_id=HF_DATA_REPO, repo_type='dataset', local_dir=data_dir)
    # snapshot_download may nest the path if it was uploaded with a prefix — flatten.
    for prefix in [os.path.join(data_dir, 'data', 'mmaps'), os.path.join(data_dir, 'mmaps')]:
        if os.path.exists(os.path.join(prefix, 'train')):
            for item in os.listdir(prefix):
                src = os.path.join(prefix, item)
                dst = os.path.join(data_dir, item)
                if not os.path.exists(dst):
                    shutil.move(src, dst)
            shutil.rmtree(os.path.join(data_dir, prefix.replace(data_dir + '/', '').split('/')[0]), ignore_errors=True)
            break
    print(f'Data downloaded \u2192 {data_dir}')
else:
    print('Data already in Drive.')

In [ ]:
# ── 5. Copy data to local SSD for fast I/O ───────────────────────────────────
import shutil, os

local_data = '/content/data/mmaps'
if not os.path.exists(f'{local_data}/train'):
    print('Copying data from Drive to local SSD (one-time per session)...')
    shutil.copytree(f'{DRIVE_DIR}/data/mmaps', local_data)
    print('Done.')
else:
    print('Local data already present.')

n_train = len([f for f in os.listdir(f'{local_data}/train') if f.endswith('.f32bin')])
print(f'Train shards: {n_train}')

In [ ]:
# ── 6. Check for existing pretrain checkpoint (auto-resume) ───────────────────
# Prefers the latest backbone_step{N}.pt over the best backbone_pretrain.pt so
# training continues from the most recent periodic save (not from the last
# best-val save, which can lag the actual stopping point by thousands of steps).
import os, re, glob

DRIVE_CKPT_DIR = f'{DRIVE_DIR}/checkpoints/pretrain'
drive_best     = f'{DRIVE_CKPT_DIR}/backbone_pretrain.pt'

def _latest_step_ckpt(ckpt_dir):
    pattern = os.path.join(ckpt_dir, 'backbone_step*.pt')
    candidates = []
    for path in glob.glob(pattern):
        m = re.search(r'backbone_step(\d+)\.pt$', path)
        if m:
            candidates.append((int(m.group(1)), path))
    if not candidates:
        return None
    return max(candidates, key=lambda x: x[0])[1]

latest_step = _latest_step_ckpt(DRIVE_CKPT_DIR)

# Fall back to HF best-checkpoint only when Drive has nothing at all.
if latest_step is None and not os.path.exists(drive_best):
    try:
        import shutil
        path = hf_hub_download(repo_id=HF_MODEL_REPO, filename='pretrain/backbone_pretrain.pt')
        shutil.copy(path, drive_best)
        print(f'Checkpoint downloaded from HF \u2192 {drive_best}')
    except Exception:
        print('No existing checkpoint on HF \u2014 starting fresh.')

if latest_step is not None:
    resume_from = latest_step
    print(f'Will resume from latest step checkpoint: {resume_from}')
elif os.path.exists(drive_best):
    resume_from = drive_best
    print(f'Will resume from best-val checkpoint: {resume_from}')
else:
    resume_from = None
    print('Starting fresh.')

In [ ]:
# ── 7. Auto-detect GPU + set hyperparameters ──────────────────────────────────
import torch, os
assert torch.cuda.is_available(), 'No GPU \u2014 enable it in Runtime \u2192 Change runtime type.'

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}  ({vram_gb:.0f} GB VRAM)')

# Pretraining encodes one asset at a time (N=1), so much larger batches than
# portfolio training.  BPTT truncation limits the recurrent state backward chain
# to K steps to reduce VRAM.  --compile cannot be combined with
# --gradient-checkpointing (crashes the recurrent GDN backbone).
if vram_gb >= 75:    # A100 80GB, H100
    BATCH_SIZE  = 128
    BPTT        = 0      # full T=64 BPTT
    GRAD_CKPT   = False
    USE_COMPILE = True   # safe: no grad_ckpt conflict
elif vram_gb >= 35:  # A100 40GB
    BATCH_SIZE  = 128
    BPTT        = 0
    GRAD_CKPT   = True
    USE_COMPILE = False
elif vram_gb >= 22:  # L4 24GB
    BATCH_SIZE  = 64
    BPTT        = 32
    GRAD_CKPT   = True
    USE_COMPILE = False
else:                # T4 16GB, V100 16GB
    BATCH_SIZE  = 32
    BPTT        = 32
    GRAD_CKPT   = True
    USE_COMPILE = False

WORKERS = min(8, os.cpu_count() or 1)
print(f'batch={BATCH_SIZE}  bptt={BPTT}  grad_ckpt={GRAD_CKPT}  compile={USE_COMPILE}  workers={WORKERS}')

In [ ]:
# ── 8. Pre-train backbone (auto-resumes from Drive) ───────────────────────────
import os, sys, subprocess

cmd = [
    sys.executable, '-u',
    '-m', 'train.pretrain',
    '--data-dir',               '/content/data/mmaps',      # local SSD \u2014 much faster than Drive
    '--symbols-file',           '/content/meridian/symbols.txt',
    '--steps',                  '20000',
    '--batch-size',             str(BATCH_SIZE),
    '--seq-len',                '64',
    '--lr',                     '3e-4',
    '--warmup-steps',           '2000',
    '--grad-clip',              '1.0',
    '--temperature',            '0.1',
    '--lambda-contrastive',     '0.7',
    '--lambda-prediction',      '0.3',
    '--fda-loss-weight',        '0.5',
    '--positive-offset',        '4',
    '--multi-asset-corr-threshold', '0.6',
    '--negative-min-separation',    '50',
    '--log-every',              '50',
    '--eval-every',             '1000',
    '--save-every',             '5000',
    '--checkpoint-dir',         DRIVE_CKPT_DIR,             # save directly to Drive
    '--workers',                str(WORKERS),
]
if resume_from:
    cmd += ['--resume-from', resume_from]
if BPTT > 0:
    cmd += ['--bptt-horizon', str(BPTT)]
if GRAD_CKPT:
    cmd.append('--gradient-checkpointing')
if USE_COMPILE:
    cmd.append('--compile')

env = os.environ.copy()
env['PYTHONPATH']       = '/content/meridian'
env['PYTHONUNBUFFERED'] = '1'

print('Running:', ' '.join(cmd))
print('-' * 80)

process = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
process.wait()

if process.returncode != 0:
    raise RuntimeError(f'Pre-training failed with return code {process.returncode}')
print('Pre-training complete.')

In [ ]:
# ── 9. Upload best checkpoint to HuggingFace ─────────────────────────────────
import os
from huggingface_hub import HfApi

best_ckpt = f'{DRIVE_CKPT_DIR}/backbone_pretrain.pt'
if not os.path.exists(best_ckpt):
    print('No best checkpoint found \u2014 nothing to upload.')
else:
    api = HfApi()
    api.upload_file(
        path_or_fileobj=best_ckpt,
        path_in_repo='pretrain/backbone_pretrain.pt',
        repo_id=HF_MODEL_REPO,
        repo_type='model',
    )
    print(f'Uploaded \u2192 {HF_MODEL_REPO}/pretrain/backbone_pretrain.pt')